# 02 受入番号集計
元ノートから受入_時刻 データの統合処理のみを抽出。

In [ ]:
import pandas as pd, glob, os
csv_dir = '/works/data/受入_時刻'
if not os.path.isdir(csv_dir):
    print('[WARN] directory not found:', csv_dir)
else:
    csv_files = glob.glob(os.path.join(csv_dir,'*.csv'))
    if not csv_files:
        print('[WARN] no csv files')
    else:
        dfs=[]
        for p in csv_files:
            df_tmp = pd.read_csv(p)
            df_tmp['伝票日付'] = pd.to_datetime(df_tmp['伝票日付'].str.replace(r'\(.*?\)','', regex=True).str.strip(), format='%Y/%m/%d')
            df_tmp['正味重量'] = df_tmp['正味重量'].replace({',':''}, regex=True).astype(float)
            df_tmp['受入番号'] = df_tmp['受入番号'].fillna(-1).astype(int)
            dfs.append(df_tmp)
        if dfs:
            df_ukeire = pd.concat(dfs, ignore_index=True)[['伝票日付','品名','正味重量','受入番号']]
            df_count = df_ukeire.groupby(['伝票日付','品名'])['受入番号'].nunique().reset_index().rename(columns={'受入番号':'台数'})
            df_merged = pd.merge(df_ukeire, df_count, on=['伝票日付','品名'], how='left').rename(columns={'台数':'搬入済台数'})
            print('[INFO] merged shape', df_merged.shape)
            df_merged.head()
        else:
            print('[WARN] no merged dfs')

: 

次: 03 モデル実行 (予約/天気) へ。